# 宏观风险配置方法复现
## 基于风险平价和风险最小化的大类资产配置

**研报来源**: 国泰君安《宏观风险配置方法思考：以风险平价和风险最小化为例》
**大类资产配置量化模型研究系列之八 (2024.05.29)**

本notebook复现研报中的三大策略:
1. **资产风险平价** (Asset Risk Parity) - 基准策略
2. **宏观风险平价** (Macro Risk Parity) - 核心策略1
3. **宏观风险最小化** (Macro Risk Minimization) - 核心策略2

In [ ]:
import sys
sys.path.insert(0, r'd:\\Documents\\trae_projects\\macro_factor_risk_parity_minimization')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('All imports successful!')

## 1. 配置与数据加载

In [ ]:
from source.config import (
    BACKTEST_START, BACKTEST_END, ASSETS, FACTOR_NAMES,
    ASSET_CLASSES, ASSET_CLASS_WEIGHT_CAP, OUTPUT_DIR
)
from source.data_loader import init_tushare, load_all_asset_prices, load_macro_indicators
from source.backtest import MacroRiskBacktester
from source.plotting import plot_all

print(f'回测区间: {BACKTEST_START} 至 {BACKTEST_END}')
print(f'资产列表: {list(ASSETS.keys())}')
print(f'宏观因子: {FACTOR_NAMES}')

## 2. 数据加载

In [ ]:
init_tushare()
print('Tushare initialized successfully!')

In [ ]:
bt = MacroRiskBacktester(start=BACKTEST_START, end=BACKTEST_END)
bt.load_data(use_cache=True)

In [ ]:
bt.asset_returns.head()

In [ ]:
print('\n资产收益率描述性统计:')
bt.asset_returns.describe()

## 3. 构建宏观因子

In [ ]:
bt.build_factors(use_pca=True)

In [ ]:
print('\n宏观因子收益率样例:')
bt.factor_returns.head(10)

In [ ]:
print('\n宏观因子相关性矩阵:')
bt.factor_returns.corr().round(3)

## 4. 准备因子数据 (协方差矩阵、暴露度)

In [ ]:
bt.prepare_factor_data(lookback=36)

## 5. 运行三大策略回测

In [ ]:
bt.run_all_strategies()

## 6. 策略对比

In [ ]:
bt.compare_all_strategies()

## 7. 详细绩效指标

In [ ]:
metrics_df = pd.DataFrame({
    name: data['metrics'] 
    for name, data in bt.strategy_results.items()
}).T

display_cols = ['annualized_return', 'annualized_volatility', 
                'max_drawdown', 'sharpe_ratio', 
                'calmar_ratio', 'monthly_turnover']
metrics_df[display_cols].round(4)

## 8. 可视化

In [ ]:
plot_all(bt, save_dir=OUTPUT_DIR)

In [ ]:
from IPython.display import Image, display

plot_files = [
    'cumulative_returns.png',
    'frc_boxplot.png', 
    'weight_allocation.png',
    'monthly_returns_heatmap.png'
]

for f in plot_files:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        print(f'=== {f} ===')
        display(Image(filename=path, width=900))
        print()

## 9. 宏观风险贡献分析

In [ ]:
macro_strategies = ['Macro Risk Parity', 'Macro Risk Minimization']

print('各策略宏观因子风险贡献率 (时序均值):\n')
for name in macro_strategies:
    if 'frc' in bt.strategy_results[name]:
        frc = bt.strategy_results[name]['frc']
        mean_frc = frc.mean() * 100
        print(f'{name}:')
        for fac, val in mean_frc.items():
            print(f'  {fac}: {val:.1f}%')
        print()

In [ ]:
asset_rp = bt.strategy_results['Asset Risk Parity']
macro_rp = bt.strategy_results['Macro Risk Parity']
macro_rm = bt.strategy_results['Macro Risk Minimization']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, data) in enumerate([
    ('Asset Risk Parity', asset_rp),
    ('Macro Risk Parity', macro_rp),
    ('Macro Risk Minimization', macro_rm)
]):
    weights = data['weights']
    avg_w = weights.mean().sort_values(ascending=True)
    avg_w = avg_w[avg_w > 0.001]
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(avg_w)))
    axes[i].barh(range(len(avg_w)), avg_w.values * 100, color=colors)
    axes[i].set_yticks(range(len(avg_w)))
    axes[i].set_yticklabels(avg_w.index, fontsize=9)
    axes[i].set_xlabel('Average Weight (%)')
    axes[i].set_title(f'{name}\nAverage Asset Weights', fontsize=12)
    for j, (bar, val) in enumerate(zip(axes[i].patches, avg_w.values)):
        axes[i].text(val * 100 + 0.1, j, f'{val:.2%}', 
                     va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'weight_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Weight comparison saved.')

## 10. 保存结果

In [ ]:
bt.save_results()
print('\n所有图表已保存到:', OUTPUT_DIR)
print('\n项目复现完成!')

---
## 重要数据说明

本项目尝试从以下开源数据源实时拉取真实市场数据:
- **tushare** (自定义API端点): 沪深300、中证1000、中证转债、国内债券指数
- **yfinance**: 标普500、纳斯达克、布伦特原油、COMEX黄金
- **akshare**: 南华商品指数等

**宏观因子数据的重要限制**:

研报使用的部分宏观数据（包括但不限于）目前无法通过免费数据源获取:

| 所需数据 | 来源说明 |
|---|---|
| PMI同比差分 | 需要官方PMI数据，akshare可能受限 |
| CPI同比、PPI同比 | 需要统计局数据，akshare/tushare可能受限 |
| 10年期国债收益率 | 可通过tushare/akshare获取部分数据 |
| 信用利差 (3年AA中票-国开债) | 需要Wind/iFinD |
| M2同比、社融同比 | tushare可能受限 |
| CRB工业原料指数 | 需要Wind/Bloomberg |
| 南华沪铜 | 需要南华期货API |
| 猪肉价格 | akshare可获取部分数据 |

**当前替代方案**:
- 使用PCA从资产收益率中提取宏观因子（作为逼近）
- 使用可获取的资产收益率模拟因子收益（mimicking portfolio近似）
- 实际复现时，部分绩效指标可能与研报存在差异

**如需精确复现研报结果**, 建议补充:
1. Wind终端数据
2. iFinD (同花顺) 数据
3. 宏观因子历史数据库